In [26]:
print("Hypothesis Test")

print("H0: μ1 = μ2 = μ3 (All group means are equal)")
print("H1: At least one group mean is different")

print("Where:")
print("μ1 = No milk drinking")
print("μ2 = Occasional milk drinking")
print("μ3 = Frequent milk drinking")

HYPOTHESIS TEST
H0: μ1 = μ2 = μ3 (All group means are equal)
H1: At least one group mean is different
Where:
μ1 = No milk drinking
μ2 = Occasional milk drinking
μ3 = Frequent milk drinking


In [10]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

df = pd.read_csv("../data/processed/YRBS_Cleaned_and_Recoded.csv")
df["NoMilkDrinking"] = df["NoMilkDrinking"].astype("category")

model = ols(
    "HowTallAreYouWithoutShoesInMeters ~ NoMilkDrinking",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

In [24]:
anova_core = anova_table.copy()

anova_core = anova_core.rename(
    columns={
        "sum_sq": "SS",
        "df": "df",
        "F": "F-value",
        "PR(>F)": "p-value"
    }
)

anova_core["MS"] = anova_core["SS"] / anova_core["df"]

# =====================================
# p-value formatting (<0.001)
# =====================================

def format_p(p):
    if p < 0.001:
        return "<0.001"
    else:
        return round(p, 4)

anova_core["p-value"] = anova_core["p-value"].apply(format_p)

# =====================================
# Reorder columns
# =====================================

anova_core = anova_core[
    ["SS", "df", "MS", "F-value", "p-value"]
]

display(anova_core.round(6))

,SS,df,MS,F-value,p-value
NoMilkDrinking,2.801031,2.0,1.400516,139.055874,<0.001
Residual,127.556868,12665.0,0.010072,NaN,NaN


In [19]:
# =====================================
# p-value + Effect Size
# =====================================

import numpy as np

# ========= p-value =========
p_value = anova_table["PR(>F)"].iloc[0]

print("=====================================")
print("STATISTICAL TEST RESULT")
print("=====================================")

print(f"p-value = {p_value:.10e}")

if p_value < 0.001:
    print("Result: Highly significant (p < 0.001)")
elif p_value < 0.05:
    print("Result: Significant (p < 0.05)")
else:
    print("Result: Not significant")

# ========= Effect Size =========
# =====================================
# EFFECT SIZE
# =====================================

ss_between = anova_table["sum_sq"].iloc[0]
ss_total = anova_table["sum_sq"].sum()

eta_sq = ss_between / ss_total

print("\n=====================================")
print("EFFECT SIZE")
print("=====================================")

print(f"Eta Squared (η²) = {eta_sq:.6f}")

# 解釋（報告用）
if eta_sq < 0.01:
    interpretation = "Very small effect"
elif eta_sq < 0.06:
    interpretation = "Small effect"
elif eta_sq < 0.14:
    interpretation = "Medium effect"
else:
    interpretation = "Large effect"

print(f"Interpretation: = The effect size was {eta_sq:.4f}, indicating a {interpretation.lower()}.")

STATISTICAL TEST RESULT
p-value = 1.8294438231e-60
Result: Highly significant (p < 0.001)

EFFECT SIZE
Eta Squared (η²) = 0.021487
Interpretation: = The effect size was 0.0215, indicating a small effect.


Interpretation:
Although the ANOVA test showed a statistically significant difference between groups (p < 0.001), the effect size was small (η² ≈ 0.02), indicating that the practical difference in height across milk consumption groups is limited.